%md
## Análise de Dados

### Pergunta 1 — Correlação e defasagem (curto prazo, jan/2024-abr/2026)

In [0]:
%sql
SELECT corr(selic_mensal_pct, rj_requeridas) AS correlacao_sem_defasagem
FROM mvp_juros_rj.gold.fato_indicadores_mensais;

Correlação com defasagem (lag), usando LAG() do SQL

In [0]:
%sql
WITH base AS (
  SELECT
    mes_referencia,
    rj_requeridas,
    LAG(selic_mensal_pct, 1) OVER (ORDER BY mes_referencia) AS selic_lag1,
    LAG(selic_mensal_pct, 2) OVER (ORDER BY mes_referencia) AS selic_lag2,
    LAG(selic_mensal_pct, 3) OVER (ORDER BY mes_referencia) AS selic_lag3,
    LAG(selic_mensal_pct, 6) OVER (ORDER BY mes_referencia) AS selic_lag6
  FROM mvp_juros_rj.gold.fato_indicadores_mensais
)
SELECT
  corr(selic_lag1, rj_requeridas) AS corr_lag1,
  corr(selic_lag2, rj_requeridas) AS corr_lag2,
  corr(selic_lag3, rj_requeridas) AS corr_lag3,
  corr(selic_lag6, rj_requeridas) AS corr_lag6
FROM base;

%md
**Visualização das séries**

Para complementar os coeficientes de correlação calculados acima,
visualização das duas séries ao longo do tempo, evidenciando o
descompasso entre a tendência suave da Selic e a maior volatilidade
mensal dos pedidos de RJ.

In [0]:
%sql
SELECT mes_referencia, selic_mensal_pct, rj_requeridas
FROM mvp_juros_rj.gold.fato_indicadores_mensais
ORDER BY mes_referencia;

Dentro da janela de dados com metodologia consistente (jan/2024 a
abr/2026), a correlação entre a Selic mensal e os pedidos de RJ é
fraca em todos os cenários testados: sem defasagem (r de
aproximadamente 0,21), com defasagem de 1 mês (r de 0,01), 2 meses
(0,04), 3 meses (-0,01) e 6 meses (-0,22, na direção oposta à
esperada). Com apenas 28 observações, nenhum desses coeficientes chega
perto de ser estatisticamente significativo.

Isso sugere que o efeito da Selic sobre RJ, se existir, não aparece
como uma resposta rápida mês a mês dentro de um período de 28 meses. A
próxima pergunta investiga se esse efeito aparece numa escala de tempo
mais longa.

---

### Pergunta 2: Padrão histórico (2016-2024, dados de contexto)

Esta análise usa a fato_contexto_anual, construída a partir de dados
institucionais anuais, na metodologia anterior à atualização de 2026
do indicador Serasa Experian. Ela foi mantida separada de propósito da
tabela mensal usada na Pergunta 1 (a justificativa está na seção de
Modelagem).

In [0]:
%sql
SELECT
  CASE
    WHEN ano BETWEEN 2016 AND 2018 THEN '1. Juros altos (2016-2018)'
    WHEN ano BETWEEN 2019 AND 2021 THEN '2. Juros baixos (2019-2021)'
    WHEN ano BETWEEN 2022 AND 2024 THEN '3. Juros altos (2022-2024)'
  END AS periodo,
  ROUND(AVG(pedidos_rj), 0)        AS media_pedidos_rj,
  ROUND(AVG(selic_media_anual), 2) AS selic_media
FROM mvp_juros_rj.gold.fato_contexto_anual
GROUP BY 1
ORDER BY 1;

%md
**Visualização do padrão histórico**

Gráfico combinando volume anual de pedidos de RJ (barras) e Selic média
anual (linha), 2016-2024, evidenciando visualmente o padrão cíclico em
"U" discutido na análise: RJ acompanha o movimento da Selic entre os
dois ciclos de alta (2016-2018 e 2022-2024) e o intervalo de juros
baixos entre eles (2019-2021).

In [0]:
%sql
SELECT ano, pedidos_rj, selic_media_anual
FROM mvp_juros_rj.gold.fato_contexto_anual
ORDER BY ano;

**Resultado**: o padrão em três períodos revela um formato em "U". No
primeiro bloco de juros altos (2016-2018, Selic média de 10,21%), o
volume médio anual de RJ foi de 1.564. No período intermediário de
juros baixos (2019-2021, Selic média de 4,38%), essa média caiu para
1.152. Já no bloco mais recente de juros altos (2022-2024, Selic média
de 12,16%), o volume voltou a subir, chegando a 1.504, um aumento de
30,6% em relação ao período de juros baixos.

Esse padrão cíclico, em que RJ sobe e desce acompanhando o mesmo
movimento da Selic, reforça a hipótese de que juros sustentadamente
elevados por vários anos acabam se refletindo em mais recuperações
judiciais. Não é possível afirmar causalidade com os dados disponíveis
(as limitações estão descritas no objetivo do projeto), mas o padrão é
consistente. Um detalhe importante: a resposta não é imediata. O ano
de 2022, mesmo já com Selic elevada em 12,43%, teve o menor volume de
RJ de toda a série, apenas 833 pedidos. O aumento expressivo só
aparece em 2023 e 2024, o que reforça a ideia de uma defasagem de 1 a
2 anos entre o início do ciclo de alta e a resposta em RJ. O mesmo
parece valer no sentido inverso: a queda de RJ observada em 2019-2021
acontece depois da queda de juros que já vinha desde 2017.

### Síntese geral

As duas perguntas de negócio, analisadas em escalas de tempo
diferentes, contam histórias complementares. No curto prazo (mês a
mês, entre 2024 e 2026), não há evidência de correlação forte entre
Selic e RJ; os coeficientes calculados ficaram fracos ou próximos de
zero em todos os lags testados. Já no longo prazo (comparando três
períodos de três anos, de 2016 a 2024), aparece um padrão cíclico em
"U": RJ acompanha o movimento da Selic subindo, descendo e subindo de
novo, o que é consistente com a hipótese de um efeito acumulado de
juros elevados, com uma defasagem de 1 a 2 anos entre o início do
ciclo e a resposta em RJ.

Esses dois resultados podem parecer contraditórios à primeira vista,
mas na verdade são o próprio achado deste projeto: se o efeito da
Selic sobre recuperações judiciais existe, ele parece operar numa
escala de tempo mais longa do que meses. Por isso não aparece num
teste de curto prazo, mesmo estando presente quando se compara ciclos
de vários anos.

A principal limitação deste projeto é não conseguir testar essa
defasagem de longo prazo com o mesmo rigor estatístico usado na
análise mensal, já que a fonte de dados anual segue uma metodologia
diferente e tem granularidade mais grosseira. Um trabalho futuro
poderia buscar uma fonte de dados judiciais mais granular e de longo
prazo, como o CNJ/DataJud, e assim testar essa defasagem com mais
precisão, unificando as duas análises numa única série temporal
contínua.